# AERIS - Tuberculose Cloud (DenseNet121, 320x320)

- Pool commun : `tawsifurrahman/tuberculosis-tb-chest-xray-dataset` (4200 images)
- Split deterministe 80/10/10 via manifests embarques (graine 42), identique a TB Edge
- Transfer learning ImageNet (DenseNet121), entree 320x320, puis fine-tune partiel
- Exports : modele `.keras`, metriques, predictions test (logits + probas)
- Attribution dataset : IEEE Access 2020, DOI 10.1109/ACCESS.2020.3031384


In [ ]:
import hashlib
import json
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (320, 320)
BATCH = 16
EPOCHS_CONGELATION = 60
EPOCHS_AJUSTEMENT = 25
COUCHES_LIBREES = 30
EPS = 1e-7

WORK = Path('/kaggle/working')
RACINE_INPUT = Path('/kaggle/input/datasets')


def trouver_dossier_images(nom_attendu):
    for base in [RACINE_INPUT, Path('/kaggle/input'), Path('/kaggle/working'), Path('.')]:
        if not base.exists():
            continue
        for candidat in sorted(base.glob(f'**/{nom_attendu}')):
            if candidat.is_dir():
                return candidat
    raise FileNotFoundError(nom_attendu)


DATA = trouver_dossier_images('TB_Chest_Radiography_Database')
print('pool TB', DATA)
MANIFEST_DIR = WORK / 'manifests'
MANIFEST_DIR.mkdir(exist_ok=True, parents=True)
ART = WORK / 'artifacts_tb_cloud_d121'
ART.mkdir(exist_ok=True, parents=True)

print('tensorflow', tf.__version__)


In [ ]:
LOTS = ['tuberculose_entrainement.csv', 'tuberculose_validation.csv', 'tuberculose_test.csv']

candidats = []
for racine in [Path('.'), WORK, RACINE_INPUT, Path('/kaggle/input')]:
    try:
        candidats.extend(racine.glob('**/tuberculose*.csv'))
    except OSError:
        pass

for nom in LOTS:
    trouve = next((c for c in candidats if c.name == nom), None)
    if trouve is None:
        raise FileNotFoundError(nom)
    pd.read_csv(trouve).to_csv(MANIFEST_DIR / nom, index=False)
    print('manifeste', nom, '->', trouve)

manifestes = {nom: pd.read_csv(MANIFEST_DIR / nom) for nom in LOTS}
for nom, df in manifestes.items():
    print(nom, len(df), 'positifs', int(df['etiquette'].sum()))

CLASSES_ATTENDUES = ('Normal', 'Tuberculosis')
dossiers_classe = {}
for classe in CLASSES_ATTENDUES:
    trouves = sorted(p for p in DATA.glob('**/' + classe) if p.is_dir())
    if not trouves:
        raise FileNotFoundError(f'classe {classe} absente de {DATA}')
    dossiers_classe[classe] = trouves[0]
    print('classe', classe, '->', dossiers_classe[classe])

index_images = {}
for classe, dossier in dossiers_classe.items():
    for motif in ('*.png', '*.jpeg', '*.jpg', '*.PNG', '*.JPEG'):
        for chemin in dossier.glob(motif):
            index_images[f'{classe}/{chemin.name}'] = chemin
print('images indexees', len(index_images))

for df in manifestes.values():
    chemins = []
    for relatif in df['chemin_relatif']:
        cle = '/'.join(str(relatif).split('/')[-2:])
        if cle not in index_images:
            raise FileNotFoundError(f'image absente du pool : {relatif}')
        chemins.append(str(index_images[cle]))
    df['chemin'] = chemins

print('toutes les images du manifeste sont resolues')


In [ ]:
entrainement = manifestes['tuberculose_entrainement.csv']
validation = manifestes['tuberculose_validation.csv']
test = manifestes['tuberculose_test.csv']


def generer(df):
    for chemin, etiquette in zip(df['chemin'], df['etiquette']):
        image = tf.keras.utils.load_img(chemin, target_size=IMG_SIZE, color_mode='rgb')
        tableau = tf.keras.applications.densenet.preprocess_input(
            tf.keras.utils.img_to_array(image)
        )
        yield tableau, float(etiquette)


AUTOTUNE = tf.data.AUTOTUNE
signature = (
    tf.TensorSpec(shape=(320, 320, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.float32),
)
ds_entrainement = (
    tf.data.Dataset.from_generator(lambda: generer(entrainement), output_signature=signature)
    .shuffle(len(entrainement), seed=SEED)
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)
ds_validation = (
    tf.data.Dataset.from_generator(lambda: generer(validation), output_signature=signature)
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)
ds_test = (
    tf.data.Dataset.from_generator(lambda: generer(test), output_signature=signature)
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)
print('lots pret pour entrainement')


In [ ]:
base = tf.keras.applications.DenseNet121(
    include_top=False, input_shape=(*IMG_SIZE, 3), weights='imagenet'
)
base.trainable = False

sortie = base.output
sortie = tf.keras.layers.GlobalAveragePooling2D()(sortie)
sortie = tf.keras.layers.Dropout(0.2)(sortie)
sortie = tf.keras.layers.Dense(64, activation='relu')(sortie)
tete = tf.keras.layers.Dense(1, activation='sigmoid', dtype='float32')(sortie)

modele = tf.keras.Model(base.input, tete)
modele.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)
rappels = [
    tf.keras.callbacks.EarlyStopping(
        patience=12, restore_best_weights=True, monitor='val_loss'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        patience=6, factor=0.5, monitor='val_loss'
    ),
]
modele.summary()


In [ ]:
congelation = modele.fit(
    ds_entrainement,
    validation_data=ds_validation,
    epochs=EPOCHS_CONGELATION,
    callbacks=rappels,
    verbose=1,
)
print('epochs_congelation', len(congelation.history['loss']))


In [ ]:
base.trainable = True
for couche in base.layers[:-COUCHES_LIBREES]:
    couche.trainable = False

modele.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)
ajustement = modele.fit(
    ds_entrainement,
    validation_data=ds_validation,
    epochs=EPOCHS_AJUSTEMENT,
    callbacks=rappels,
    verbose=1,
)
print('epochs_ajustement', len(ajustement.history['loss']))


In [ ]:
y_true = test['etiquette'].to_numpy(dtype=float)
probabilites = modele.predict(ds_test, verbose=0).reshape(-1)
bornes = np.clip(probabilites, EPS, 1.0 - EPS)
logits = np.log(bornes / (1.0 - bornes))
saturation = int(np.sum(probabilites >= 1.0 - EPS))

pd.DataFrame(
    {
        'chemin_relatif': test['chemin_relatif'],
        'etiquette': y_true,
        'proba_tb': probabilites,
        'logit_tb': logits,
    }
).to_csv(ART / 'predictions_test_tb_cloud_d121.csv', index=False)

metriques = {
    'mode': 'tb_cloud_densenet121',
    'auc': float(roc_auc_score(y_true, probabilites)),
    'brier': float(brier_score_loss(y_true, probabilites)),
    'nll': float(log_loss(y_true, probabilites)),
    'nb_test': int(len(test)),
    'positifs_test': int(y_true.sum()),
    'saturation_float32': saturation,
    'epochs_congelation': len(congelation.history['loss']),
    'epochs_ajustement': len(ajustement.history['loss']),
}
(ART / 'metrics_tb_cloud_d121.json').write_text(json.dumps(metriques, indent=2))
modele.save(ART / 'aeris_tb_cloud_densenet121.keras')
print('OK', metriques)
